# Week 3 · Class 2 — Exploring wearable data with Plotly 📈⌚

**Goal of today (60 min):** answer real questions about a dataset with the *right* chart, and make those charts look professional.

We work with a dataset like the ones health platforms collect from wearables:
**250 users × 365 days = 91,250 rows** of daily steps, sleep, heart rate, HRV and stress, from 8 European cities and 4 devices.

| Part | Question we ask | Charts |
|---|---|---|
| 0 | What does the data look like? Is it clean? | missing-values bar, box |
| 1 | How are values distributed? | histogram, violin, ECDF |
| 2 | How do groups compare? | bar (sorted, grouped, highlighted) |
| 3 | Are variables related? | scatter + trendline, scatter matrix, heatmap |
| 4 | How do things change over time? | line, rolling mean, range slider, calendar heatmap |
| 5 | What is the composition? | sunburst, treemap |
| 6 | Where? | bubble map |
| 7 | How does it evolve? | animation |
| 8 | Make it presentable | templates, annotations, subplots |
| 9 | From notebook to app | wrap a chart in a function |

> 💡 Rule of the day: **first the question, then the chart.**

## ⚙️ Setup

📁 Put `wearables_daily.csv` next to this notebook (in **Colab**: drag it into the Files panel on the left), then run the two cells below.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# In Colab charts render automatically. If they don't show in VS Code/Jupyter, uncomment:
# import plotly.io as pio; pio.renderers.default = "notebook"

In [4]:
df = pd.read_csv("../data/wearables_daily.csv", parse_dates=["date"])   # parse_dates: read 'date' as a real date
df.head()

,user_id,date,city,device,age,gender,steps,active_minutes,sleep_hours,deep_sleep_pct,resting_hr,hrv_ms,stress_score,calories,age_group,lat,lon
0,U001,2025-10-01,Berlin,Apple Watch,58,female,11208,91.0,6.24,12.8,68.0,38.7,56.0,2412.0,50-59,52.52,13.4
1,U001,2025-10-02,Berlin,Apple Watch,58,female,14360,123.0,5.86,17.7,65.0,38.6,56.0,2666.0,50-59,52.52,13.4
2,U001,2025-10-03,Berlin,Apple Watch,58,female,10323,81.0,6.95,15.6,65.0,31.5,56.0,2337.0,50-59,52.52,13.4
3,U001,2025-10-04,Berlin,Apple Watch,58,female,10048,87.0,7.06,24.8,65.0,30.5,41.0,2350.0,50-59,52.52,13.4
4,U001,2025-10-05,Berlin,Apple Watch,58,female,10675,98.0,6.76,12.8,64.0,39.0,59.0,2419.0,50-59,52.52,13.4


## 0. First look: what do we have? Is it clean? 🔍

Before plotting anything, we always check **size, types, missing values and weird values**.

In [5]:
print(df.shape)
df.info()

(91250, 17)
<class 'pandas.DataFrame'>
RangeIndex: 91250 entries, 0 to 91249
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   user_id         91250 non-null  str           
 1   date            91250 non-null  datetime64[us]
 2   city            91250 non-null  str           
 3   device          91250 non-null  str           
 4   age             91250 non-null  int64         
 5   gender          91250 non-null  str           
 6   steps           91250 non-null  int64         
 7   active_minutes  91250 non-null  float64       
 8   sleep_hours     88514 non-null  float64       
 9   deep_sleep_pct  88514 non-null  float64       
 10  resting_hr      91250 non-null  float64       
 11  hrv_ms          83017 non-null  float64       
 12  stress_score    91250 non-null  float64       
 13  calories        91250 non-null  float64       
 14  age_group       91250 non-null  str           
 15  l

In [6]:
df.describe().round(1)

,date,age,steps,active_minutes,sleep_hours,deep_sleep_pct,resting_hr,hrv_ms,stress_score,calories,lat,lon
count,91250,91250.0,91250.0,91250.0,88514.0,88514.0,91250.0,83017.0,91250.0,91250.0,91250.0,91250.0
mean,2026-04-01 00:00:00,44.4,8376.5,69.4,7.2,17.6,71.7,44.3,50.5,2210.9,47.8,5.4
min,2025-10-01 00:00:00,20.0,1179.0,0.0,4.2,5.0,44.0,10.0,6.0,1659.0,38.7,-9.1
25%,2025-12-31 00:00:00,32.0,5671.0,47.0,6.7,14.7,68.0,34.7,43.0,2015.0,40.4,-3.7
50%,2026-04-01 00:00:00,44.0,7572.0,64.0,7.2,17.6,72.0,44.2,51.0,2156.0,48.9,4.9
75%,2026-07-01 00:00:00,57.0,10121.0,85.0,7.8,20.4,75.0,53.9,58.0,2344.0,52.4,13.4
max,2026-09-30 00:00:00,69.0,94771.0,387.0,10.9,35.0,90.0,92.2,100.0,5017.0,55.7,16.4
std,NaN,14.4,4213.4,32.8,0.8,4.3,5.2,13.1,11.4,281.2,5.5,8.5


👀 Look at `steps`: the median is ~7,500 but the **max is above 90,000**. Nobody walks 90k steps a day — that smells like a sensor glitch.

Let's **see** the missing values first. A bar chart of `isna().sum()` is the quickest data-quality chart.

In [7]:
missing = df.isna().mean().mul(100).round(1).sort_values()
missing = missing[missing > 0].reset_index()
missing.columns = ["column", "pct_missing"]

fig = px.bar(missing, x="pct_missing", y="column", orientation="h", text="pct_missing",
             title="Which columns have missing values? (%)")
fig.show()

Is the missing HRV random, or does it depend on the device? 🤔

In [8]:
hrv_missing = df.groupby("device")["hrv_ms"].apply(lambda s: s.isna().mean() * 100).reset_index()
px.bar(hrv_missing, x="device", y="hrv_ms", title="% of days with missing HRV, by device",
       labels={"hrv_ms": "% missing"})

💡 **Insight:** missing HRV is *not random* — it comes from one device. Good to know before averaging HRV per device!

Now the outliers. A **box plot** shows them instantly as isolated points.

In [9]:
px.box(df, x="steps", title="Steps per day — spot the glitches on the right")

In [10]:
# Clean: remove impossible values and keep a clean copy
df = df[df["steps"] < 50_000].copy()
df["month"] = df["date"].dt.to_period("M").astype(str)
df["weekday"] = df["date"].dt.day_name()
print(df.shape)

(91190, 19)


## 1. Distributions: what values are typical? 📊

| Chart | When to use |
|---|---|
| Histogram | shape of one numeric variable |
| Histogram + `marginal` | shape **and** outliers in one figure |
| Overlaid histograms / violins | compare the shape between groups |
| ECDF | "what % of days are below X?" |

In [11]:
px.histogram(df, x="sleep_hours", nbins=60, marginal="box",
             title="How long do people sleep?")

In [12]:
px.histogram(df, x="steps", color="device", barmode="overlay", opacity=0.6,
             histnorm="probability density", nbins=80,
             title="Steps distribution per device")

`histnorm='probability density'` lets us compare groups of **different sizes** fairly.

In [13]:
age_order = ["20-29", "30-39", "40-49", "50-59", "60+"]
px.violin(df, x="age_group", y="resting_hr", box=True, color="age_group",
          category_orders={"age_group": age_order},
          title="Resting heart rate by age group")

In [14]:
px.ecdf(df, x="steps", color="gender",
        title="What share of days are below X steps?")

💡 Read the ECDF at 10,000 steps: what % of days do **not** reach the famous 10k goal?

## 2. Comparing groups 🏆

Bar charts are the workhorse. Three tricks make them much better:
1. **Sort** the bars.
2. Go **horizontal** when labels are long or many.
3. **Highlight** what matters, grey out the rest.

In [15]:
by_city = df.groupby("city", as_index=False)["steps"].mean().sort_values("steps")
px.bar(by_city, x="steps", y="city", orientation="h", text_auto=".0f",
       title="Average daily steps per city")

In [16]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = df.groupby(["weekday", "device"], as_index=False)["sleep_hours"].mean()
px.bar(by_day, x="weekday", y="sleep_hours", color="device", barmode="group",
       category_orders={"weekday": day_order},
       title="Sleep by weekday and device")

😴 Weekends are visible. But the y-axis starts at 0, so the differences look tiny — is that the right choice here? Bars **must** start at 0. If you want to zoom in on differences, use dots or lines instead.

In [17]:
# Highlight one category: Lisbon in colour, the rest in grey
by_city["highlight"] = np.where(by_city["city"] == "Lisbon", "Lisbon", "Other cities")
px.bar(by_city, x="steps", y="city", orientation="h", color="highlight",
       color_discrete_map={"Lisbon": "#E4572E", "Other cities": "lightgrey"},
       title="How does Lisbon compare?")

## 3. Relationships between variables 🔗

With 90k rows a raw scatter becomes a blob. **Aggregate first** (one point per user) — a very common EDA move.

In [18]:
per_user = df.groupby(["user_id", "age_group", "device", "gender"], as_index=False)[
    ["steps", "sleep_hours", "resting_hr", "hrv_ms", "stress_score", "age"]].mean()
per_user.head()

,user_id,age_group,device,gender,steps,sleep_hours,resting_hr,hrv_ms,stress_score,age
0,U001,50-59,Apple Watch,female,10765.358904,6.378719,66.339726,32.345479,54.709589,58.0
1,U002,60+,Fitbit,male,14368.309589,8.094628,62.745205,31.736986,41.989041,65.0
2,U003,40-49,Withings,female,6835.189041,7.930084,74.498630,47.422477,46.361644,43.0
3,U004,60+,Garmin,female,6098.445055,6.465620,77.049451,29.910165,56.271978,62.0
4,U005,50-59,Garmin,female,3487.712329,7.101289,78.128767,38.926849,53.572603,51.0


In [19]:
px.scatter(per_user, x="steps", y="resting_hr", trendline="ols",
           hover_name="user_id", hover_data=["age_group", "device"],
           title="More steps → lower resting heart rate?")

`trendline='ols'` fits a linear regression (needs `statsmodels`). Hover on the line to see the equation and R².

In [20]:
px.scatter(per_user, x="age", y="hrv_ms", color="gender", trendline="ols",
           marginal_y="box", title="HRV decreases with age")

In [21]:
px.scatter_matrix(per_user, dimensions=["steps", "sleep_hours", "resting_hr", "hrv_ms", "stress_score"],
                  color="age_group", opacity=0.6, height=750,
                  title="All pairs at once")

In [22]:
corr = df[["steps", "active_minutes", "sleep_hours", "deep_sleep_pct",
           "resting_hr", "hrv_ms", "stress_score", "age"]].corr().round(2)

px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
          title="Correlation matrix")

💡 Correlations go from −1 to +1, so we use a **diverging** colour scale centred at 0 (remember the colour-scale lesson!).

## 4. Time series ⏳

Daily data is noisy. A **rolling mean** shows the trend; a **range slider** lets the reader zoom.

In [23]:
daily = df.groupby("date", as_index=False)["steps"].mean()
daily["7-day mean"] = daily["steps"].rolling(7).mean()

fig = px.line(daily, x="date", y=["steps", "7-day mean"], title="Average steps per day")
fig.update_xaxes(rangeslider_visible=True)
fig.show()

☀️ A clear **summer peak** and a **weekly zig-zag** (weekends). Let's look at the weekly pattern with a calendar heatmap.

In [24]:
daily_sleep = df.groupby("date", as_index=False)["sleep_hours"].mean()
daily_sleep["week"] = daily_sleep["date"].dt.strftime("%G-W%V")
daily_sleep["weekday"] = daily_sleep["date"].dt.day_name().str[:3]

calendar = daily_sleep.pivot(index="weekday", columns="week", values="sleep_hours")
calendar = calendar.loc[["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]]

px.imshow(calendar, color_continuous_scale="Blues", aspect="auto",
          title="Average sleep — calendar view", labels={"color": "hours"})

In [25]:
# Compare several cities over time, one small chart each (facets)
monthly = df.groupby(["month", "city"], as_index=False)["steps"].mean()
px.line(monthly, x="month", y="steps", facet_col="city", facet_col_wrap=4, height=500,
        title="Monthly steps per city")

## 5. Part-to-whole 🥧➡️🌞

Pie charts fail with many slices. For **hierarchies** use a sunburst or a treemap.

In [ ]:
users = df.drop_duplicates("user_id")
px.sunburst(users, path=["city", "device"], title="Users per city and device (click to zoom!)")

In [ ]:
px.treemap(per_user, path=["device", "age_group"], values="steps", color="sleep_hours",
           color_continuous_scale="Viridis",
           title="Total steps by device and age (colour = sleep)")

## 6. Geographical data 🗺️

`px.scatter_map` draws on an open map — **no token needed** (it replaces the old `scatter_mapbox`).

In [ ]:
per_city = df.groupby(["city", "lat", "lon"], as_index=False).agg(
    users=("user_id", "nunique"), steps=("steps", "mean"), sleep=("sleep_hours", "mean"))

px.scatter_map(per_city, lat="lat", lon="lon", size="users", color="steps",
               hover_name="city", hover_data={"sleep": ":.2f", "lat": False, "lon": False},
               size_max=40, zoom=3, map_style="carto-positron",
               color_continuous_scale="Tealgrn", title="Users and activity per city")

## 7. Animations 🎬

An animation adds the **time** dimension to a scatter. Press ▶️.

In [ ]:
city_month = df.groupby(["month", "city"], as_index=False).agg(
    steps=("steps", "mean"), sleep=("sleep_hours", "mean"), users=("user_id", "nunique"))

px.scatter(city_month, x="steps", y="sleep", size="users", color="city", text="city",
           animation_frame="month", range_x=[6500, 10500], range_y=[6.9, 7.65],
           size_max=45, title="Cities through the year: steps vs sleep")

⚠️ Always fix `range_x` / `range_y` in animations, otherwise the axes jump at every frame.

## 8. From exploratory to presentation-ready ✨

Exploratory charts are fast and ugly; presentation charts must be **understood in 3 seconds**.
Tools: **templates**, clear **titles and labels**, **annotations**, **reference lines**.

In [ ]:
import plotly.io as pio
print(list(pio.templates))

In [ ]:
fig = px.line(daily, x="date", y="7-day mean", template="plotly_dark",
              labels={"7-day mean": "Steps (7-day average)", "date": ""},
              title="<b>Europeans walk ~30% more in summer than in winter</b><br>"
                    "<sup>Average daily steps, 250 users, Oct 2025 – Sep 2026</sup>")
fig.update_traces(line_color="#00E5A8", line_width=3)

peak = daily.loc[daily["7-day mean"].idxmax()]
fig.add_annotation(x=peak["date"], y=peak["7-day mean"], text="Summer peak",
                   showarrow=True, arrowhead=2, ay=-40)
fig.add_hline(y=10_000, line_dash="dot", line_color="grey",
              annotation_text="10k goal", annotation_position="bottom right")
fig.update_layout(font_family="Arial", hovermode="x unified")
fig.show()

💡 The **title tells the insight**, the subtitle tells the data. Now let's combine several charts with `make_subplots` — a mini dashboard!

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=(
    "Steps over time", "Sleep distribution", "Steps per device", "Resting HR vs steps"))

fig.add_trace(go.Scatter(x=daily["date"], y=daily["7-day mean"], mode="lines"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["sleep_hours"], nbinsx=40), row=1, col=2)
fig.add_trace(go.Box(x=df["device"], y=df["steps"], boxpoints=False), row=2, col=1)
fig.add_trace(go.Scatter(x=per_user["steps"], y=per_user["resting_hr"], mode="markers",
                         marker=dict(opacity=0.6)), row=2, col=2)

fig.update_layout(height=700, showlegend=False, template="plotly_dark",
                  title="Wearable health — overview")
fig.show()

It works… but it is static. To let users **choose** the city, device or metric, we need an **app**. That is Dash — next part!

## 9. From notebook to app: wrap charts in functions 🧩

In the project, every chart will be a **function**: data goes in, figure comes out.
This makes it reusable — the app will call it every time a user changes a filter.

In [ ]:
def metric_by_device(data, metric="steps", template="plotly_dark"):
    """Box plot of any metric for each device."""
    fig = px.box(data, x="device", y=metric, color="device", points=False,
                 template=template, title=f"{metric} by device")
    fig.update_layout(showlegend=False)
    return fig


metric_by_device(df, "sleep_hours")

In [ ]:
# Same function, different inputs → the core idea of a dashboard
berlin = df[df["city"] == "Berlin"]
metric_by_device(berlin, "resting_hr", template="plotly_white")

## 🎯 Recap

- **Question first**, chart second.
- EDA starts with **quality checks**: missing values (are they random?) and outliers.
- Distributions → histogram/violin/ECDF · Comparisons → sorted bars · Relationships → scatter + trendline, heatmap · Time → rolling mean · Hierarchy → sunburst/treemap · Space → map.
- **Aggregate** big data before plotting.
- Presentation charts: insight as title, labels, annotations, consistent template.
- Charts in **functions** = ready for a dashboard. 🚀

### 📚 To go further
- [Plotly Express reference](https://plotly.com/python-api-reference/plotly.express.html)
- [Built-in colour scales](https://plotly.com/python/builtin-colorscales/)
- [Templates & theming](https://plotly.com/python/templates/)
- [Dash tutorial](https://dash.plotly.com/tutorial)